## Regression RNN: Sunspots

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from RNNUtils import train_model, plot_errors
import matplotlib.pyplot as plt 
import matplotlib.dates as mdates

### Load and Prepare the Data

The sunspot data can be obtained from Kaggle at [sunspots](https://www.kaggle.com/datasets/robervalt/sunspots/). This is a CSV file with the average total sunspots each month since records began in 1749.

In [ ]:
sunspots_file = 'Sunspots.csv'

# Load the dataset
df = pd.read_csv(sunspots_file, index_col=0)

# Assuming the columns are named 'Index', 'Date', 'Sunspots'
df['Date'] = pd.to_datetime(df['Date'])
df.sort_values('Date', inplace=True)

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
df.head()

In [ ]:
# Split the dataset into training and test sets
train_df, test_df = train_test_split(df, test_size=0.2, shuffle=False)

# Scale the data
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df[['Monthly Mean Total Sunspot Number']])
test_scaled = scaler.transform(test_df[['Monthly Mean Total Sunspot Number']])

The data is a single time series and this needs to be broken down into sections of the sequence of length `training_win` followed by a section of target with length `target_win`. The following function generates the sequences and corresponding targets using a sliding window approach.

In [ ]:
def create_sequences(data, training_win, target_win):
    """
    Create sequences of `training_win` length data with `target_win` length targets.
    """
    sequences = []
    targets = []

    for i in range(len(data) - training_win - target_win + 1):
        sequences.append(data[i:(i + training_win)])
        targets.append(data[(i + training_win):(i + training_win + target_win)])

    return np.array(sequences), np.array(targets)

Training and test dataloaders can then be created. Note that the data is not shuffles to preserve the order during training.

In [ ]:
def prepare_dataloaders(data, training_win, target_win, batch_size, device, drop_last=True):
    """
    Prepare PyTorch DataLoaders for training and test data.
    """
    # Create sequences
    sequences, targets = create_sequences(data, training_win, target_win)

    # Convert to PyTorch tensors
    sequences_tensor = torch.tensor(sequences.astype(np.float32)).to(device)
    targets_tensor = torch.tensor(targets.astype(np.float32)).to(device)

    # Create TensorDataset
    dataset = TensorDataset(sequences_tensor, targets_tensor)

    # Create DataLoader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=drop_last)

    return dataloader

In [ ]:
training_win = 12  # length of the training window
target_win = 1   # length of the target window
hidden_size = 64
batch_size = 32   # size of each mini-batch
lr = 0.0001 # learning rate
epochs = 100

In [ ]:
train_dataloader = prepare_dataloaders(train_scaled, training_win, target_win, batch_size, device=device)
test_dataloader = prepare_dataloaders(test_scaled, training_win, target_win, batch_size, device=device)

### Define the RNN Model

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Assuming x is of shape (batch_size, seq_length, input_size)
        out, _ = self.rnn(x)
        # Take the output of the last time step
        out = out[:, -1, :]
        out = self.linear(out)
        return out

In [ ]:
model = SimpleRNN(input_size=target_win, hidden_size=hidden_size, output_size=target_win).to(device)
model

### Train Model

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
train_errors, test_errors = train_model(model, train_dataloader, test_dataloader, 
                                        loss_fn, optimizer, epochs)

In [ ]:
filename = 'rnn_sunspot_loss.png'

In [ ]:
plot_errors(train_errors, test_errors , filename)

In [ ]:
full_dataset = scaler.transform(df[['Monthly Mean Total Sunspot Number']])
full_dataset.shape

In [ ]:
full_dataloader = prepare_dataloaders(full_dataset, training_win, target_win, 
                                      batch_size, device=device, drop_last=False)

In [ ]:
def getpredandtruth(model, dataloader):
    predictions = []
    ground_truth = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            # Generate predictions
            outputs = model(inputs)
            predictions.extend(outputs.detach().cpu().numpy())
            ground_truth.extend(targets.detach().cpu().numpy())

    # Convert lists to numpy arrays for plotting
    predictions = np.array(predictions).flatten()
    ground_truth = np.array(ground_truth).flatten()
    return predictions, ground_truth

In [ ]:
predictions, ground_truth = getpredandtruth(model, full_dataloader)

In [ ]:
def plot_predandtruth(dates, ground_truth, predictions, test_date, filename):
    plt.figure(figsize=(20, 10))
    plt.plot(dates, ground_truth, color='gray', linestyle='-', label='Ground Truth')
    plt.plot(dates, predictions, color='black', linestyle='-', label='Predictions')

    # Add a black dashed vertical line at test_date
    plt.axvline(x=test_date, color='black', linestyle='--', label='Train / Test Split')

    plt.gca().xaxis.set_major_locator(mdates.YearLocator(25))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.gca().tick_params(axis='both', which='major', labelsize=14)
    plt.xlabel('Time', fontsize=14)
    plt.ylabel('Values', fontsize=14)
    plt.legend(fontsize=14)
    plt.savefig(filename, dpi=300, format='png')
    plt.show()

In [ ]:
plot_predandtruth(df['Date'][12:], ground_truth, predictions, test_df['Date'].iloc[0], 'sunspot_prediction.png')